# Milestone 1: Dataset Audit & Data Contract
## Adaptive Cyber Threat Intelligence under Concept Drift

This notebook performs a comprehensive quality audit of the cleaned **CICIDS2017** dataset (`cicids2017_cleaned.csv`).
It inspects dataset metrics, evaluates class imbalance, performs quality validation, and audits feature production feasibility to prevent **training-serving skew** when deploying live network sensors.

In [ ]:
import sys
from pathlib import Path

# Add project root to sys.path
project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import pandas as pd
import IPython.display as display
from ml.src.data.audit import (
    inspect_dataset_meta,
    analyze_target_distribution,
    validate_data_quality,
    compute_numeric_stats,
    audit_feature_feasibility,
    plot_class_distribution,
    asdict
)

dataset_path = project_root / "data" / "raw" / "cicids2017" / "cicids2017_cleaned.csv"
output_dir = project_root / "artifacts" / "reports"
print(f"Project root: {project_root}")
print(f"Dataset path: {dataset_path}")

--- 
### 1. Load Dataset & Inspect Metadata

In [ ]:
df = pd.read_csv(dataset_path)
meta = inspect_dataset_meta(df, file_path=dataset_path, target_col="Attack Type")
print(f"Rows: {meta.num_rows:,}")
print(f"Columns: {meta.num_columns}")
print(f"Memory Usage: {meta.memory_usage_mb:.2f} MB")
print(f"File Size: {meta.file_size_mb:.2f} MB")
print(f"Target Present: {meta.target_column_present} ({meta.target_column_name})")

--- 
### 2. Target Class Distribution Analysis

In [ ]:
target_df = analyze_target_distribution(df, target_col="Attack Type")
display.display(target_df)

# Display saved plot
plot_path = output_dir / "class_distribution.png"
if plot_path.exists():
    display.display(display.Image(filename=str(plot_path)))

--- 
### 3. Data Quality Validation

In [ ]:
quality_report = validate_data_quality(df)
print(f"Total Rows: {quality_report['total_rows']:,}")
print(f"Duplicate Rows: {quality_report['duplicate_rows']:,} ({quality_report['duplicate_row_percentage']:.2f}%)")
print(f"Missing Values: {quality_report['total_missing_values']}")
print(f"Columns with +Inf: {list(quality_report['columns_with_pos_inf'].keys())}")
print(f"Columns with -Inf: {list(quality_report['columns_with_neg_inf'].keys())}")
print(f"Constant Columns: {quality_report['constant_columns']}")
print(f"Low Cardinality Columns: {quality_report['low_cardinality_columns']}")

--- 
### 4. Feature Production Feasibility Assessment

In [ ]:
feature_audit_df = audit_feature_feasibility(df, target_col="Attack Type")
display.display(feature_audit_df.head(15))

print("\nCategory breakdown:")
display.display(feature_audit_df['category'].value_counts())